In [73]:
import pandas as pd
import numpy as np

In [74]:
df_base = pd.read_csv('df_final.csv')

In [75]:
df = df_base

In [76]:
df

,Цена,Дата публикации,Город,is_class_eco,is_class_comfort,is_class_business,is_class_elite,is_brick,is_monolith,is_panel,...,Month_Public,DayOfWeek_Public,Floor_Ratio,Is_First_Floor,Is_Last_Floor,Area_per_Room,Infrastructure_Score,Площадь_log,Цена_log,Цена_за_квадратный_метр_log
0,5964400,2025-09-22 13:37:50,Киров,0,0,0,0,0,0,0,...,9,0,1.000000,0,1,24.000000,3.42,3.891820,15.601319,11.730126
1,5829810,2025-09-29 20:46:23,Киров,0,0,0,0,0,0,0,...,9,0,0.916667,0,0,39.000000,1.26,3.688879,15.578495,11.914940
2,6400900,2025-09-29 10:26:27,Киров,0,1,0,0,0,0,0,...,9,0,1.000000,0,1,17.333333,4.01,3.970292,15.671949,11.720714
3,5814900,2025-09-29 20:37:16,Киров,0,0,0,0,0,0,0,...,9,0,0.833333,0,0,39.000000,1.26,3.688879,15.575934,11.912379
4,6315750,2025-09-29 20:42:24,Киров,0,0,0,0,0,0,0,...,9,0,1.000000,0,1,40.000000,1.26,3.713572,15.658557,11.969684
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17504,6336000,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,1.000000,0,1,32.000000,2.70,4.174387,15.661758,11.502885
17505,6399400,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,0.666667,0,0,32.000000,2.70,4.174387,15.671715,11.512842
17506,6252400,2025-10-16 15:39:56,Гагарин,0,0,1,0,0,1,0,...,10,3,0.666667,0,0,31.500000,2.70,4.158883,15.648476,11.505351
17507,6316200,2025-10-16 15:39:50,Гагарин,0,0,1,0,0,1,0,...,10,3,0.888889,0,0,31.500000,2.70,4.158883,15.658628,11.515504


In [77]:
# Обработка признаков с длинным хвостом
dist_cols = [
    'dist_to_city_center', 'dist_to_school', 'dist_to_kindergarten', 
    'dist_to_park', 'dist_to_bus_stop', 'dist_to_supermarket'
]
for col in dist_cols:
    df[f'{col}_log'] = np.log1p(df[col])
    

In [78]:
# Удаляем утечки, лишние столбцы и целевую переменную 
cols_to_drop = dist_cols + ['Площадь', 'Дата публикации', 
                            'Цена', 'Цена_log', 'Цена_за_квадратный_метр', 'Цена_за_квадратный_метр_log']
X = df.drop(columns=cols_to_drop)
# Целевая переменная
y = df['Цена_за_квадратный_метр_log']

In [79]:
# Закодируем город и субъект
X = pd.get_dummies(X, columns=['Город', 'Субъект РФ'], drop_first=True)

In [80]:
from sklearn.model_selection import train_test_split
# Разделяем выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [81]:
from sklearn.preprocessing import StandardScaler
# Масштабирование признаков
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Линейная регрессия

In [82]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error

# Обучение
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

# Предсказание
y_pred_log = lr.predict(X_test_scaled)

# Обратное преобразование (из логарифма в рубли)
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

# Метрики
print(f"MAE: {mean_absolute_error(y_test_real, y_pred_real):.0f} руб/м²")
print(f"MAPE: {mean_absolute_percentage_error(y_test_real, y_pred_real)*100:.2f}%")
print(f"R2: {r2_score(y_test_real, y_pred_real):.4f}")

MAE: 14485 руб/м²
MAPE: 10.64%
R2: 0.7567


### Ridge Регрессия (L2 регуляризация)

In [83]:
from sklearn.linear_model import Ridge

# Обучение
ridge = Ridge(alpha=1.0) 
ridge.fit(X_train_scaled, y_train)

# Предсказание
y_pred_log = ridge.predict(X_test_scaled)

# Обратное преобразование
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

# Метрики
print(f"MAE: {mean_absolute_error(y_test_real, y_pred_real):.0f} руб/м²")
print(f"MAPE: {mean_absolute_percentage_error(y_test_real, y_pred_real)*100:.2f}%")
print(f"R2: {r2_score(y_test_real, y_pred_real):.4f}")

MAE: 14485 руб/м²
MAPE: 10.64%
R2: 0.7567


### Случайный лес

In [84]:
from sklearn.ensemble import RandomForestRegressor

# Обучение
rf = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)
rf.fit(X_train_scaled, y_train)

# Предсказание
y_pred_log = rf.predict(X_test_scaled)

# Обратное преобразование
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

# Метрики

print(f"MAE: {mean_absolute_error(y_test_real, y_pred_real):.0f} руб/м²")
print(f"MAPE: {mean_absolute_percentage_error(y_test_real, y_pred_real)*100:.2f}%")
print(f"R2: {r2_score(y_test_real, y_pred_real):.4f}")

MAE: 4899 руб/м²
MAPE: 3.55%
R2: 0.9457


После выполнения базовых моделей лучшей себя показала: Random forest

## Линейная регрессия:   

+ MAE: 14485 руб/м²   
+ MAPE: 10.64%                
+ R2: 0.7567                  

## Ridge Регрессия: 

+ MAE: 14485 руб/м²
+ MAPE: 10.64%
+ R2: 0.7567

## Случайный лес:

+ MAE: 4899 руб/м²
+ MAPE: 3.55%
+ R2: 0.9457

## Теперь сделаем кросс-валидацию и перезапустим модели

In [85]:
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.pipeline import make_pipeline

In [86]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [87]:
# Создаем пайплайн
lr_pipeline = make_pipeline(StandardScaler(), LinearRegression())


# Делаем предсказания через кросс-валидацию
y_pred_log = cross_val_predict(lr_pipeline, X, y, cv=kf)

# Защита от переполнения (если модель сошла с ума и выдала бесконечность)
y_pred_log = np.clip(y_pred_log, 0, 20) 

# Переводим логарифмы обратно в рубли
y_pred_real = np.expm1(y_pred_log)
y_true_real = np.expm1(y)

# Метрики
mae = mean_absolute_error(y_true_real, y_pred_real)
mape = mean_absolute_percentage_error(y_true_real, y_pred_real)

print(f"MAE: {mae:.0f} руб/м²")
print(f"MAPE: {mape*100:.2f}%")
print(f"R2: {r2_score(y_true_real, y_pred_real):.4f}")

MAE: 14624 руб/м²
MAPE: 10.81%
R2: 0.7539


In [88]:
# Создаем пайплайн
ridge_pipeline = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

# Предсказание
y_pred_log = cross_val_predict(ridge_pipeline, X, y, cv=kf)

# Перевод в рубли
y_pred_real = np.expm1(y_pred_log)
y_true_real = np.expm1(y)

# Метрики
mae = mean_absolute_error(y_true_real, y_pred_real)
mape = mean_absolute_percentage_error(y_true_real, y_pred_real)
r2 = cross_val_score(ridge_pipeline, X, y, cv=kf, scoring='r2').mean()

print(f"MAE: {mae:.0f} руб/м²")
print(f"MAPE: {mape*100:.2f}%")
print(f"Средний R2: {r2:.4f}")

MAE: 14624 руб/м²
MAPE: 10.81%
Средний R2: 0.7474


In [89]:
rf_model = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)

# Предсказание
y_pred_log = cross_val_predict(rf_model, X, y, cv=kf)

# Перевод в рубли
y_pred_real = np.expm1(y_pred_log)
y_true_real = np.expm1(y)

# Метрики
mae = mean_absolute_error(y_true_real, y_pred_real)
mape = mean_absolute_percentage_error(y_true_real, y_pred_real)
r2 = cross_val_score(rf_model, X, y, cv=kf, scoring='r2').mean()

print(f"MAE: {mae:.0f} руб/м²")
print(f"MAPE: {mape*100:.2f}%")
print(f"Средний R2: {r2:.4f}")

MAE: 4929 руб/м²
MAPE: 3.58%
Средний R2: 0.9451


# Значения метрик моделей после кросс-валидации: 


## Линейная регрессия:   

+ MAE: 14485 руб/м²   →    MAE: 14624 руб/м²
+ MAPE: 10.64%        →    MAPE: 10.81%    
+ R2: 0.7567          →    R2: 0.7539    

## Ridge Регрессия: 

+ MAE: 14485 руб/м²   →    MAE: 14624 руб/м²
+ MAPE: 10.64%        →    MAPE: 10.81%
+ R2: 0.7567          →    R2: 0.7474 

## Случайный лес:

+ MAE: 4899 руб/м²    →    MAE: 4929 руб/м²
+ MAPE: 3.55%         →    MAPE: 3.58%
+ R2: 0.9457          →    R2: 0.9451

# Подбор гиперпараметров

## Подбор параметров для Ridge

In [90]:
from sklearn.model_selection import GridSearchCV

In [91]:
# Сетка параметров 
param_grid_ridge = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 150.0, 200.0, 500.0]
}


ridge = Ridge()

# Переберем все варианты, используя кросс-валидацию 
#  оптимизируем ошибку
grid_ridge = GridSearchCV(ridge, param_grid_ridge, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_ridge.fit(X_train_scaled, y_train)

# Лучшая модель
best_ridge = grid_ridge.best_estimator_
print(f"Лучшие параметры: {grid_ridge.best_params_}")

# Проверка на тесте
y_pred_log = best_ridge.predict(X_test_scaled)
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

mae = mean_absolute_error(y_test_real, y_pred_real)
mape = mean_absolute_percentage_error(y_test_real, y_pred_real)
r2 = r2_score(y_test_real, y_pred_real)

print(f"\nРезультаты Ridge после настройки:")
print(f"MAE:  {mae:,.0f} руб/м²")
print(f"MAPE: {mape*100:.2f}%")
print(f"R2:   {r2:.4f}")


Лучшие параметры: {'alpha': 150.0}

Результаты Ridge после настройки:
MAE:  14,479 руб/м²
MAPE: 10.64%
R2:   0.7566


In [92]:
from sklearn.model_selection import RandomizedSearchCV

In [93]:

# Сетка гиперпараметров
param_dist_rf = {
    'n_estimators': [100, 200, 300, 500],        # Количество деревьев
    'max_depth': [None, 15, 20, 30, 40],         # Максимальная глубина
    'min_samples_split': [2, 5, 10],             # Минимум примеров для разделения узла
    'min_samples_leaf': [1, 2, 4],               # Минимум примеров в листе
    'max_features': ['sqrt', 'log2', None]       # Сколько признаков брать
}

rf = RandomForestRegressor(random_state=42)



random_search_rf = RandomizedSearchCV(
    rf, 
    param_distributions=param_dist_rf, 
    n_iter=20,           # Количество попыток
    cv=3,                # 3 фолда кросс-валидации
    scoring='neg_mean_squared_error', 
    n_jobs=-1, 
    random_state=42,
    verbose=1
)

random_search_rf.fit(X_train_scaled, y_train) 

# Лучшая модель
best_rf = random_search_rf.best_estimator_
print(f"\nЛучшие параметры: {random_search_rf.best_params_}")

# Проверка на тесте
y_pred_log = best_rf.predict(X_test_scaled)
y_pred_real = np.expm1(y_pred_log)
y_test_real = np.expm1(y_test)

mae = mean_absolute_error(y_test_real, y_pred_real)
mape = mean_absolute_percentage_error(y_test_real, y_pred_real)
r2 = r2_score(y_test_real, y_pred_real)

print(f"\nРезультаты Random Forest после настройки:")
print(f"MAE:  {mae:,.0f} руб/м²")
print(f"MAPE: {mape*100:.2f}%")
print(f"R2:   {r2:.4f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits

Лучшие параметры: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': None}

Результаты Random Forest после настройки:
MAE:  4,845 руб/м²
MAPE: 3.51%
R2:   0.9461




# Выводы:

В ходе работы была проведена подготовка данных, обучение базовых моделей, проверка их устойчивости через кросс-валидацию и финальная оптимизация гиперпараметров.



## 1. Сравнение: Обучение (Train/Test) vs Кросс-валидация (CV)

Кросс-валидация показала высокую устойчивость моделей. Различия между разовым обучением и проверкой на 5 фолдах минимальны, что говорит об отсутствии переобучения и надежности данных.

###  Линейные модели
Линейная регрессия и Ridge показали практически идентичные результаты. Это указывает на то, что базовая регуляризация (в Ridge) при текущих настройках не вносит существенных изменений по сравнению с обычным МНК.

| Метрика | Линейная Регрессия (Base → CV) | Ridge Регрессия (Base → CV) |
| :--- | :--- | :--- |
| **MAE** (руб/м²) | 14 485 → **14 624** | 14 485 → **14 624** |
| **MAPE** (%) | 10.64% → **10.81%** | 10.64% → **10.81%** |
| **R² Score** | 0.7567 → **0.7539** | 0.7567 → **0.7474** |

> Зависимость цены от признаков слишком сложная для прямой линии.

---

###  Случайный лес (Random Forest)
Безусловный лидер эксперимента. Модель демонстрирует феноменальную стабильность.

| Метрика | Случайный лес (Base → CV) |
| :--- | :--- |
| **MAE** (руб/м²) | 4 899 → **4 929** |
| **MAPE** (%) | 3.55% → **3.58%** |
| **R² Score** | 0.9457 → **0.9451** |

---

## 2. Результаты оптимизации гиперпараметров (Tuning)

После подбора параметров  нам удалось выжать дополнительную точность из моделей.

###  Итоговая таблица сравнения (Best Models)

| Модель | MAE (руб/м²) | MAPE (%) | R² Score |
| :--- | :--- | :--- | :--- |
| **Ridge (Tuned)** | 14 479 | 10.64% | 0.7566 |
| **Random Forest (Tuned)** | **4 845** | **3.51%** | **0.9461** |

**Результаты тюнинга:**
* **Ridge:** Удалось немного снизить ошибку относительно кросс-валидации (с 14 624 до 14 479), вернувшись к показателям базового обучения.
* **Random Forest:** Тюнинг позволил получить лучший результат за все время экспериментов — **MAPE 3.51%**.

---

## 3. Итоговое заключение

1.  **Победитель:** **Random Forest**.
    * Модель ошибается в среднем всего на **4 845 рублей** с квадратного метра.
    * Для квартиры площадью 50 м² средняя погрешность составит около **240 000 рублей**, что для рынка недвижимости является отличным результатом.
2.  **Качество данных:** Высокие метрики ($R^2 \approx 0.95$) подтверждают, что собранные данные (включая One-Hot кодирование городов и логарифмирование расстояний) качественные и содержат всю необходимую информацию для ценообразования.
